# Streaming Responses

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will learn how to stream agent output in real time using different stream modes.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Create an Agent

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")

agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
)

## 4. Stream Mode: updates

Returns state changes from each node in the agent graph.

In [ ]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="Explain quantum computing in 3 sentences.")]},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"--- {node_name} ---")
        for msg in update.get("messages", []):
            print(f"  [{msg.type}] {msg.content[:200]}")

## 5. Stream Mode: messages

Token-by-token streaming for responsive UIs.

In [ ]:
for message, metadata in agent.stream(
    {"messages": [HumanMessage(content="Write a haiku about programming.")]},
    stream_mode="messages",
):
    if message.content and metadata.get("langgraph_node") == "agent":
        print(message.content, end="", flush=True)
print()

## 6. Streaming with Tools

See the full execution flow: agent decision, tool call, and response.

In [ ]:
from langchain_core.tools import tool

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))

agent_with_tools = create_react_agent(
    model=model,
    tools=[calculate],
    prompt="You are a math assistant.",
)

for chunk in agent_with_tools.stream(
    {"messages": [HumanMessage(content="What's (15 + 27) * 3?")]},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"[{node_name}]")
        for msg in update.get("messages", []):
            if msg.type == "tool":
                print(f"  Tool {msg.name}: {msg.content}")
            else:
                print(f"  {msg.content[:200]}")

## 7. Custom Stream Writer from Tools

Tools can emit custom progress events using `StreamWriter`.

In [ ]:
from langgraph.types import StreamWriter

@tool
def analyze_data(query: str, writer: StreamWriter) -> str:
    """Analyze data and stream progress updates."""
    writer("Starting analysis...")
    writer(f"Processing query: {query}")
    writer("Running calculations...")
    result = f"Analysis complete for '{query}': 42 records found"
    writer("Done!")
    return result

agent_custom = create_react_agent(
    model=model,
    tools=[analyze_data],
    prompt="You are a data analyst.",
)

for chunk in agent_custom.stream(
    {"messages": [HumanMessage(content="Analyze sales data for Q4")]},
    stream_mode=["updates", "custom"],
):
    print(chunk)

## 8. Multiple Stream Modes

Combine modes by passing a list. Each chunk becomes a tuple of `(mode, data)`.

In [ ]:
for mode, chunk in agent_with_tools.stream(
    {"messages": [HumanMessage(content="Calculate 100 / 7")]},
    stream_mode=["updates", "messages"],
):
    if mode == "updates":
        for node_name, update in chunk.items():
            print(f"[Update: {node_name}]")
    elif mode == "messages":
        msg, meta = chunk
        if msg.content:
            print(f"[Token] {msg.content}", end="")
print()

## Summary

- Use `agent.stream()` for real-time output instead of `agent.invoke()`
- `stream_mode="updates"` shows step-by-step execution
- `stream_mode="messages"` enables token-by-token streaming
- `stream_mode="custom"` captures tool progress via `StreamWriter`
- Combine modes with a list for comprehensive streaming